# **S-learner**

The S-learner (Single learner) is the most straightforward way an ML model estimates casual effects (marketing campaign or discount actually causes a customer to stay)

## **The Churn Scenario Setup**

Imagine you run a subscription software company and want to know: Does offering a $20 discount code ($X$) actually prevent customers from churning ($Y$)?

In your historical data, you have:
- Treatment ($X$): $1$ if the customer received a discount, $0$ if they did not.
- Outcome ($Y$): $1$ if the customer churned (canceled), $0$ if they stayed. Because $Y$ is binary, our machine learning model will be a classifier (like XGBoost or Logistic Regression) that predicts the probability of churning ($P(Y=1)$).
- Confounders ($Z_1, Z_2$): Variables that affect both whether a customer gets a discount and whether they churn.
    - $Z_1$: Customer tenure (months subscribed).
    - $Z_2$: App logins per week.
    
(Why are $Z_1$ and $Z_2$ confounders? Because your retention team historically gave discounts mostly to low-usage, high-risk customers, creating selection bias.)

## **Implementation**

### **Training and Testing of the Model**

In [178]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,GridSearchCV, StratifiedKFold
from sklearn.metrics import  recall_score, precision_score, f1_score,roc_auc_score

**Generate synthetic data for demonstration**

In [179]:
def generate_enterprise_churn_data(n=10000, seed=42):
    np.random.seed(seed)
    
    # 1. Realistic Confounders
    # Tenure: Gamma distribution (skewed: most customers are newer, fewer are multi-year veterans)
    tenure_months = (np.clip(np.random.gamma(shape=2, scale=6, size=n), 1, 60)).astype(int)
    
    # Weekly Logins: Poisson distribution
    weekly_logins = np.random.poisson(lam=4, size=n)
    
    # Support Tickets: Higher tickets usually indicate friction
    support_tickets = np.random.poisson(lam=1.5, size=n)
    
    # Monthly Spend Tier ($10, $25, $50, $100)
    monthly_spend = np.random.choice([10, 25, 50, 100], size=n, p=[0.4, 0.3, 0.2, 0.1])
    
    # 2. Confounding Mechanism (Propensity Score)
    # Retention teams historically give discounts to customers with low tenure, low logins, AND high support tickets.
    propensity_logit = -0.5 + (0.05 * tenure_months) - (0.20 * weekly_logins) + (0.40 * support_tickets) - (0.005 * monthly_spend)
    discount_prob = 1 / (1 + np.exp(-propensity_logit))
    treatment_x = np.random.binomial(1, discount_prob)
    
    # 3. Outcome Mechanism (Churn)
    # True Causal Effect: Offering a discount reduces the log-odds of churning by -0.35 (~15-20% absolute drop)
    true_causal_effect = -0.35
    
    baseline_logit = (
        1.2 
        - (0.06 * tenure_months) 
        - (0.15 * weekly_logins) 
        + (0.50 * support_tickets) 
        - (0.01 * monthly_spend)
        + (true_causal_effect * treatment_x) # The true causal impact!
    )
    churn_prob = 1 / (1 + np.exp(-baseline_logit))
    outcome_y = np.random.binomial(1, churn_prob)
    
    df = pd.DataFrame({
        'tenure_months': tenure_months,
        'weekly_logins': weekly_logins,
        'support_tickets': support_tickets,
        'monthly_spend': monthly_spend,
        'discount_given': treatment_x, # Treatment (X)
        'churned': outcome_y           # Outcome (Y)
    })
    
    return df

In [180]:
df = generate_enterprise_churn_data(n=10000)

In [181]:
df.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,churned
0,14,7,1,25,0,0
1,8,4,3,50,0,0
2,8,9,2,10,1,1
3,8,7,0,10,0,0
4,27,3,1,100,0,1


**Split the data**

In [182]:
# train test split
X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [183]:
print(f"Dataset Split Sizes:")
print(f" - Train Set: {len(X_train)} rows")
print(f" - Test Set: {len(X_test)} rows\n")

Dataset Split Sizes:
 - Train Set: 8000 rows
 - Test Set: 2000 rows



**Train S-learner**

In [184]:
# from sklearn import metrics
# metrics.get_scorer_names()

In [ ]:
# Define Model
# https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.RandomForestClassifier.html
rf = RandomForestClassifier(random_state=50)

# Define Cross-Validation Strategy
# This takes care of train and validaition splits for us
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)

# Define Hyperparameter Grid
param_grid = {
    'n_estimators': [50,150, 250],
    'max_depth': [10, 15, 20]
}

# Define Scoring Metrics
# https://scikit-learn.org/stable/auto_examples/model_selection/plot_multi_metric_evaluation.html#
# https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers
scoring_metrics = {
    'AUC': 'roc_auc',
    'R': 'recall',
    'P': 'precision',
    'F1': 'f1'
}
# Define Grid Search
grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    scoring = scoring_metrics,
    cv = cv,
    refit = "AUC", # Refit the model based on the best AUC score
    n_jobs = -2, # Use all available cores except one
    verbose = 3
)

In [ ]:
grid

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=50)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 15, ...], 'n_estimators': [50, 150, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'AUC': 'roc_auc', 'F1': 'f1', 'P': 'precision', 'R': 'recall'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'AUC'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time

In [187]:
print("Starting Grid Search...")
grid.fit(X_train, y_train)
print("Grid Search Completed.")

Starting Grid Search...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END max_depth=10, n_estimators=50; AUC: (test=0.701) F1: (test=0.679) P: (test=0.659) R: (test=0.700) total time=   1.5s
[CV 2/5] END max_depth=10, n_estimators=50; AUC: (test=0.710) F1: (test=0.686) P: (test=0.661) R: (test=0.713) total time=   1.7s
[CV 3/5] END max_depth=10, n_estimators=50; AUC: (test=0.701) F1: (test=0.686) P: (test=0.645) R: (test=0.733) total time=   0.9s
[CV 4/5] END max_depth=10, n_estimators=50; AUC: (test=0.687) F1: (test=0.686) P: (test=0.653) R: (test=0.723) total time=   1.1s
[CV 5/5] END max_depth=10, n_estimators=50; AUC: (test=0.704) F1: (test=0.683) P: (test=0.670) R: (test=0.697) total time=   1.0s
[CV 1/5] END max_depth=10, n_estimators=150; AUC: (test=0.702) F1: (test=0.681) P: (test=0.656) R: (test=0.707) total time=   1.6s
[CV 2/5] END max_depth=10, n_estimators=150; AUC: (test=0.711) F1: (test=0.683) P: (test=0.664) R: (test=0.702) total time=   1.7s
[CV 

In [188]:
print(f"\nBest Estimator: {grid.best_estimator_}\n")
print(f"Best Cross-Validation ROC AUC: {np.round(grid.best_score_, 2)}")


Best Estimator: RandomForestClassifier(max_depth=10, n_estimators=250, random_state=50)

Best Cross-Validation ROC AUC: 0.7


**Evaluate on test data**

In [189]:
best_model = grid.best_estimator_

In [190]:
# The 1 means we are taking the probability of the positive class (churn = 1)
test_prediction_proba = best_model.predict_proba(X_test)[:, 1]
test_prediction = best_model.predict(X_test)

In [191]:
# Evaluations
test_auc = roc_auc_score(y_test, test_prediction_proba)
test_recall = recall_score(y_test, test_prediction)
test_precision = precision_score(y_test, test_prediction)
test_f1 = f1_score(y_test, test_prediction)

In [192]:
print("Test Set Evaluation Metrics:")
print(f"ROC-AUC: {np.round(test_auc, 2)}")
print(f"Recall: {np.round(test_recall, 2)}")
print(f"Precision: {np.round(test_precision, 2)}")
print(f"F1-Score: {np.round(test_f1, 2)}")

Test Set Evaluation Metrics:
ROC-AUC: 0.71
Recall: 0.72
Precision: 0.67
F1-Score: 0.69


### **Counterfactuals (ATE): S-Learner Approach**

So the model has been trained and tested. Now we Simulate A/B groups 

In [193]:
# Group A: Discount Given
test_x1 = X_test.copy()
test_x1['discount_given'] = 1
prob_y1 = best_model.predict_proba(test_x1)[:, 1]

In [194]:
# Group B: No Discount Given
test_x2 = X_test.copy()
test_x2['discount_given'] = 0
prob_y2 = best_model.predict_proba(test_x2)[:, 1]

In [195]:
# Calculate Average Treatment Effect (ATE)
ate = np.round(np.mean(prob_y1 - prob_y2),3)
print(f"Average Treatment Effect (ATE) of Discount on Churn Probability: \n{ate}")

Average Treatment Effect (ATE) of Discount on Churn Probability: 
-0.058


**Inspection of Prediction group A**

In [196]:
test_x1['predicted_churn'] = best_model.predict(test_x1)

In [198]:
test_x1['actual_discount'] = X_test['discount_given']
test_x1['actual_churn'] = y_test
test_x1.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn,actual_discount,actual_churn
9102,35,6,1,25,1,0,0,0
7868,5,2,2,50,1,1,0,1
4176,5,2,0,100,1,0,0,0
4161,4,2,1,10,1,1,1,1
8770,6,4,1,25,1,0,0,1


**Inspection of Prediction group B**

In [199]:
test_x2['predicted_churn'] = best_model.predict(test_x2)

In [201]:
test_x2['actual_discount'] = X_test['discount_given']
test_x2['actual_churn'] = y_test
test_x2.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn,actual_discount,actual_churn
9102,35,6,1,25,0,0,0,0
7868,5,2,2,50,0,1,0,1
4176,5,2,0,100,0,0,0,0
4161,4,2,1,10,0,1,1,1
8770,6,4,1,25,0,1,0,1


### **Interpretation of ATE**

Since estimated Average Treatment Effect (ATE) is **-0.058** (or roughly **-5.8%**), it means:

> **Offering the discount causes an absolute 5.8% reduction in the probability of customer churn**, on average, after properly controlling for all confounders (like tenure, app logins, support tickets, and monthly spend).

So what does it actually mean?

1. **Direction (Negative):** Because churn represents "leaving" ($1 = \text{churned}, 0 = \text{stayed}$), a **negative** causal effect is a **good thing**. It means the discount is working; it pushes the probability of canceling downward.
2. **Magnitude (5.8 Percentage Points):** It is an *absolute percentage point change*, not a relative percentage. For example:
    - If a customer's baseline predicted churn risk without a discount was **20.0%**, giving them the discount drops their predicted churn risk to **14.2%** ($20.0\% - 5.8\%$).

3. **Causally Isolated:** Because the S-learner (backed by the machine learning model and cross-validation) controlled for selection bias, this isn't just a coincidence or correlation. It proves that the retention team's discount program actively saved customers who might have otherwise left.